# Lembar Kerja Mahasiswa (LKM)
## Latihan Mandiri — Pertemuan 08: Regresi Linear (Bagian 3, Sudut Pandang Probabilistik)

Lembar kerja ini menuntun Anda mengerjakan **kedelapan soal Latihan Mandiri** pada bagian Penutup modul `Pertemuan08_Regresi_Linear_3_Probabilistik.ipynb`, yang mencakup ill-conditioning, kesetaraan kuadrat terkecil dengan MLE, selang kepercayaan, dan teori keputusan.

---

### Identitas Mahasiswa

*Klik dua kali sel ini untuk mengedit, isi titik-titik di bawah, lalu tekan `Shift + Enter`.*

| | |
|---|---|
| **Nama** | ............................................................ |
| **NIM** | ............................................................ |
| **Kelas / Rombel** | ............................................................ |
| **Nama Dosen** | ............................................................ |
| **Tanggal Pengumpulan** | ............................................................ |

---

### Capaian yang Diukur

Setelah menyelesaikan lembar kerja ini, Anda diharapkan mampu:

1. Menunjukkan pengaruh derajat kolinearitas dan kekuatan regularisasi terhadap kestabilan bobot regresi.
2. Membuktikan secara empiris sifat tak bias penduga kuadrat terkecil dan kaitannya dengan hukum bilangan besar.
3. Menghitung dan menafsirkan selang kepercayaan untuk satu pengamatan maupun rata-rata beberapa pengamatan.
4. Mengenali kapan selang kepercayaan menjadi tidak jujur akibat overfitting.
5. Membandingkan ketangguhan fungsi kerugian MSE dan MAE terhadap pencilan.
6. Menjelaskan kesetaraan antara maksimisasi log-likelihood dan minimisasi kuadrat galat.


---
## Petunjuk Pengerjaan

**Baca bagian ini sebelum mulai.**

1. Lembar kerja ini **berdiri sendiri** (*self-contained*): bagian Persiapan membangun ulang seluruh fungsi dan variabel yang dipakai kedelapan latihan.
2. Tetap dianjurkan membuka `Pertemuan08_Regresi_Linear_3_Probabilistik.ipynb` di jendela terpisah sebagai rujukan penjelasan konsepnya.
3. Jalankan sel kode **berurutan dari atas ke bawah**.
4. Sel yang berisi tanda `____` atau komentar `# TODO` **harus Anda lengkapi sendiri**.
5. Setiap latihan memiliki **tabel hasil pengamatan** dan **kotak jawaban** markdown. Klik dua kali untuk mengedit, lalu tekan `Shift + Enter`.

> **Penting.** Nilai `random_state` diturunkan dari NIM Anda, sehingga angka hasil percobaan setiap mahasiswa akan sedikit berbeda. Pola dan kesimpulannya akan tetap sama.

### Perkiraan waktu

| Bagian | Perkiraan |
|---|---|
| Persiapan | 5 menit |
| Latihan 1–2 (ill-conditioning) | 25 menit |
| Latihan 3–4 (MLE dan hukum bilangan besar) | 25 menit |
| Latihan 5–7 (selang kepercayaan dan ketangguhan) | 35 menit |
| Latihan 8 (refleksi konseptual) | 15 menit |
| Kesimpulan dan ekspor PDF | 10 menit |

### Cara menyimpan sebagai PDF

* **JupyterLab:** `File` → `Save and Export Notebook As...` → `HTML`, lalu buka berkas HTML di browser dan cetak (`Ctrl + P`) dengan tujuan **Save as PDF**.
* **Google Colab:** `File` → `Print` → tujuan **Save as PDF**.
* Beri nama berkas: `LKM_Latihan08_NIM_NamaLengkap.pdf`


---
## Persiapan

Bagian ini membangun ulang fungsi dan variabel dari notebook utama. **Sudah lengkap — jalankan saja**, kecuali sel P.2 yang meminta NIM Anda.

### P.1 Memuat pustaka

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as st
from scipy.optimize import linprog
from sklearn.preprocessing import PolynomialFeatures

warnings.filterwarnings("ignore")
print("Semua pustaka berhasil dimuat.")

### P.2 Mengisi NIM Anda

In [ ]:
# TODO (a): ganti dengan tiga digit terakhir NIM Anda, misalnya 137
NIM_3_DIGIT = ____

SEED = int(NIM_3_DIGIT)
print("random_state pribadi Anda (SEED) =", SEED)

### P.3 Membangun Ulang: Fungsi Ill-Conditioning (Bagian 1)

Fungsi berikut membangun matriks rancangan dengan dua fitur yang hampir kolinear (seperti pada Bagian 1 notebook utama), lalu membandingkan bobot dari **persamaan normal** dan dari **Ridge**.

In [ ]:
W_BENAR_1 = np.array([3.0, 2.0, -1.0])

def analisis_illconditioning(gangguan, lam, seed=SEED):
    """Membangun dua fitur hampir kolinear, membandingkan persamaan normal dan Ridge.

    Argumen:
        gangguan: seberapa jauh x2 menyimpang dari x1 (kecil = sangat kolinear).
        lam     : kekuatan regularisasi Ridge.
    """
    rng = np.random.default_rng(seed)
    N, D = 500, 2
    X = np.ones((N, D + 1))
    x1 = rng.standard_normal(N)
    x2 = x1 + gangguan * rng.standard_normal(N)
    X[:, 1], X[:, 2] = x1, x2
    t = 3 + 2 * x1 - 1 * x2 + 0.1 * rng.standard_normal(N)

    XtX = X.T @ X
    Xty = X.T @ t
    w_ne = np.linalg.solve(XtX, Xty)

    _, s, _ = np.linalg.svd(X, full_matrices=False)
    kappa2 = (s.max() / s.min()) ** 2

    w_ridge = np.linalg.solve(XtX + lam * np.eye(D + 1), Xty)
    return kappa2, w_ne, w_ridge


k, wn, wr = analisis_illconditioning(1e-4, 1e-2)
print("Uji cepat -> cond(XtX) =", round(k, 2), " | w_ne =", np.round(wn, 2), " | w_ridge =", np.round(wr, 2))

### P.4 Membangun Ulang: Simulasi Monte Carlo (Bagian 2)

Fungsi berikut mengulang pelatihan regresi linear pada **banyak dataset acak** yang dibangkitkan dari model yang sama, untuk memeriksa sifat tak bias penduga kuadrat terkecil.

In [ ]:
def simulasi_ols(sigma, M, seed=SEED):
    """Melatih OLS pada M dataset acak, mengembalikan seluruh bobot hasilnya.

    Argumen:
        sigma: simpangan baku derau.
        M    : banyaknya dataset yang disimulasikan.
    """
    rng = np.random.default_rng(seed)
    n = 80
    x = np.linspace(0, 5, n)
    X = np.c_[x, np.ones_like(x)]
    w_star = np.array([2.0, 1.0])
    X_pinv = np.linalg.pinv(X)  # dihitung sekali saja, dipakai berulang (vektorisasi)

    ws = np.empty((M, 2))
    for m in range(M):
        y = X @ w_star + rng.normal(0, sigma, size=n)
        ws[m] = X_pinv @ y
    return ws, w_star


ws_uji, w_star_uji = simulasi_ols(sigma=1.0, M=200)
print("Uji cepat (M=200) -> rata-rata w:", np.round(ws_uji.mean(axis=0), 3), " | w* =", w_star_uji)

### P.5 Membangun Ulang: Data dan Model untuk Selang Kepercayaan (Bagian 3)

Tiga fungsi berikut memisahkan tahap **membangkitkan data**, **mencocokkan polinomial**, dan **menghitung lebar pita kepercayaan** — sehingga masing-masing dapat dipakai ulang secara independen pada Latihan 5, 6, dan 7.

> **Catatan.** Latihan 5–7 memakai `SEED_DEMO` yang **tetap** (bukan SEED pribadi Anda), karena ketiganya bersifat demonstratif dan perlu menunjukkan pola yang sama ke semua mahasiswa.

In [ ]:
SEED_DEMO = 29


def bangun_data_sinus(N=80, sigma_derau=0.25, seed=SEED_DEMO):
    rng = np.random.default_rng(seed)
    x = rng.uniform(-1, 1, (N, 1))
    t = np.sin(3 * np.pi * x) + sigma_derau * rng.normal(size=(N, 1))
    return x, t


def cocokkan_polinomial(x, t, derajat):
    """Regresi kuadrat terkecil biasa (MSE / L2) memakai basis polinomial."""
    phi = PolynomialFeatures(derajat, include_bias=True)
    X = phi.fit_transform(x)
    w = np.linalg.lstsq(X, t, rcond=None)[0]
    return phi, w


def cocokkan_polinomial_l1(x, t, derajat):
    """Regresi galat mutlak (MAE / L1) memakai pemrograman linear.

    Regresi L1 tidak punya rumus tertutup seperti L2, sehingga diselesaikan
    sebagai persoalan optimisasi linear: meminimalkan total |residual|.
    """
    phi = PolynomialFeatures(derajat, include_bias=True)
    X = phi.fit_transform(x)
    n, p = X.shape
    tt = t.ravel()
    c = np.concatenate([np.zeros(2 * p), np.ones(2 * n)])
    A_eq = np.hstack([X, -X, np.eye(n), -np.eye(n)])
    bounds = [(0, None)] * (2 * p + 2 * n)
    hasil = linprog(c, A_eq=A_eq, b_eq=tt, bounds=bounds, method="highs")
    w = (hasil.x[:p] - hasil.x[p:2 * p]).reshape(-1, 1)
    return phi, w


x_demo, t_demo = bangun_data_sinus()
print("Data siap:", x_demo.shape, t_demo.shape)
print("Fungsi siap: cocokkan_polinomial(x, t, derajat), cocokkan_polinomial_l1(x, t, derajat)")

---
# Latihan Mandiri 1 — Mengubah Derajat Kolinearitas

**Soal asli:** *Pada Bagian 1, ubah gangguan `1e-4` menjadi `1e-1` sehingga kedua fitur tidak lagi hampir identik. Bagaimana bilangan kondisinya berubah? Apakah bobot dari persamaan normal menjadi wajar kembali?*

### 1.1 Membandingkan dua tingkat gangguan

Lengkapi bagian bertanda `____`. Nilai `lam` tetap $10^{-2}$ seperti pada notebook utama.

In [ ]:
for gangguan in [1e-4, ____]:
    kappa2, w_ne, w_ridge = analisis_illconditioning(gangguan, lam=1e-2)
    print(f"gangguan = {gangguan:.0e}")
    print(f"  cond(XtX)             = {kappa2:.4e}")
    print(f"  w dari persamaan normal = {np.round(w_ne, 3)}")
    print(f"  w dari Ridge             = {np.round(w_ridge, 3)}")
    print(f"  (bobot sebenarnya: {W_BENAR_1})\n")

### 1.2 Tabel Hasil Pengamatan

*Bobot sebenarnya adalah $[3,\ 2,\ -1]$.*

| Gangguan | Bilangan kondisi cond($\mathbb{X}^T\mathbb{X}$) | $\mathbf{w}$ dari persamaan normal |
|---|---|---|
| $10^{-4}$ | ...... | ...... |
| $10^{-1}$ | ...... | ...... |

### 1.3 Pertanyaan Analisis

**1a.** Berapa kali lipat bilangan kondisi berubah saat gangguan diperbesar dari $10^{-4}$ menjadi $10^{-1}$?

> *Jawaban Anda:*
>
> ......

**1b.** Apakah bobot dari persamaan normal pada gangguan $10^{-1}$ sudah mendekati $[3,2,-1]$? Bandingkan dengan hasil pada gangguan $10^{-4}$.

> *Jawaban Anda:*
>
> ......

**1c.** Mengapa memperbesar gangguan (membuat fitur **kurang** kolinear) menyelesaikan masalah ini, padahal kita tidak mengubah `lam` sama sekali?

> *Jawaban Anda:*
>
> ......


---
# Latihan Mandiri 2 — Mencari λ Terbaik untuk Ridge

**Soal asli:** *Masih di Bagian 1, coba beberapa nilai `lam`: `1e-6`, `1e-2`, dan `1`. Berapa nilai yang menghasilkan bobot paling dekat dengan nilai sebenarnya $[3, 2, -1]$?*

### 2.1 Mencoba tiga nilai λ

Lengkapi bagian bertanda `____`. Gangguan ditahan tetap $10^{-4}$ (kolinearitas sedang, seperti nilai bawaan di Bagian 1).

In [ ]:
hasil_lam = []
for lam in [1e-6, 1e-2, 1]:
    # TODO (a): panggil analisis_illconditioning dengan gangguan=1e-4 dan lam ini
    kappa2, w_ne, w_ridge = analisis_illconditioning(gangguan=____, lam=lam)

    # TODO (b): hitung jarak (norma Euclidean) antara w_ridge dan W_BENAR_1
    jarak = np.linalg.norm(w_ridge - ____)

    hasil_lam.append({"lam": lam, "w_ridge": np.round(w_ridge, 3), "jarak_ke_benar": round(jarak, 4)})
    print(f"lam={lam:<8}  w_ridge={np.round(w_ridge, 3)}  jarak={jarak:.4f}")

lam_terbaik = min(hasil_lam, key=lambda h: h["jarak_ke_benar"])["lam"]
print("\nλ dengan bobot PALING DEKAT ke [3,2,-1]:", lam_terbaik)

### 2.2 Tabel Hasil Pengamatan

| λ | $\mathbf{w}_{\text{ridge}}$ | Jarak ke $[3,2,-1]$ |
|---|---|---|
| $10^{-6}$ | ...... | ...... |
| $10^{-2}$ | ...... | ...... |
| $1$ | ...... | ...... |

### 2.3 Pertanyaan Analisis

**2a.** λ mana yang menghasilkan bobot paling dekat dengan $[3,2,-1]$ pada percobaan Anda?

> *Jawaban Anda:*
>
> ......

**2b.** Pada λ = $10^{-6}$ (regularisasi nyaris tidak ada), apakah bobotnya masih menunjukkan gejala ill-conditioning seperti pada persamaan normal murni? Jelaskan.

> *Jawaban Anda:*
>
> ......

**2c.** Pada λ = 1 (regularisasi kuat), apakah bobotnya menjadi lebih stabil tetapi justru lebih jauh dari $[3,2,-1]$ dibanding λ = $10^{-2}$? Simpulkan pertukaran (*trade-off*) yang terjadi antara ketiga nilai λ ini.

> *Jawaban Anda:*
>
> ......


---
# Latihan Mandiri 3 — Pengaruh σ terhadap Sebaran Penduga

**Soal asli:** *Pada Bagian 2, ubah `sigma` dari 1,0 menjadi 0,1 lalu 3,0. Bagaimana pengaruhnya terhadap sebaran $\hat{\mathbf{w}}$ pada simulasi Monte Carlo? Apakah rata-ratanya tetap mendekati $\mathbf{w}^*$?*

### 3.1 Menjalankan simulasi untuk tiga nilai σ

Lengkapi bagian bertanda `____`. Kita memakai $M=500$ dataset agar prosesnya tetap cepat.

In [ ]:
hasil_sigma = []
for sigma in [0.1, 1.0, 3.0]:
    # TODO (a): jalankan simulasi_ols dengan sigma ini dan M=500
    ws, w_star = simulasi_ols(sigma=____, M=500)

    rata2 = ws.mean(axis=0)
    simpangan = ws.std(axis=0)
    hasil_sigma.append({"sigma": sigma, "rata2_w": np.round(rata2, 4), "std_w": np.round(simpangan, 4)})
    print(f"sigma={sigma}  rata-rata w = {np.round(rata2, 4)}  simpangan baku w = {np.round(simpangan, 4)}  (w*={w_star})")

### 3.2 Menggambar sebaran $\hat{w}_1$ (kemiringan) untuk ketiga σ

In [ ]:
plt.figure(figsize=(7, 4))
for sigma in [0.1, 1.0, 3.0]:
    ws, _ = simulasi_ols(sigma=sigma, M=500)
    plt.hist(ws[:, 0], bins=30, alpha=0.5, label=f"σ = {sigma}", density=True)
plt.axvline(2.0, color="black", ls="--", label="w* (kemiringan sebenarnya)")
plt.xlabel("Nilai ŵ₁ (kemiringan)"); plt.ylabel("Kepadatan")
plt.title("Sebaran Penduga Kemiringan untuk Berbagai σ")
plt.legend(); plt.show()

### 3.3 Tabel Hasil Pengamatan

| σ | Rata-rata $\hat{\mathbf{w}}$ | Simpangan baku $\hat{\mathbf{w}}$ |
|---|---|---|
| 0,1 | ...... | ...... |
| 1,0 | ...... | ...... |
| 3,0 | ...... | ...... |

### 3.4 Pertanyaan Analisis

**3a.** Bagaimana simpangan baku $\hat{\mathbf{w}}$ berubah seiring σ membesar? Apakah perubahannya kira-kira sebanding (linear) dengan σ?

> *Jawaban Anda:*
>
> ......

**3b.** Apakah rata-rata $\hat{\mathbf{w}}$ tetap mendekati $\mathbf{w}^* = [2,1]$ pada **ketiga** nilai σ, termasuk saat σ = 3,0 (derau besar)?

> *Jawaban Anda:*
>
> ......

**3c.** Jelaskan mengapa **sifat tak bias** (jawaban 3b) dan **besarnya sebaran** (jawaban 3a) adalah dua hal yang berbeda. Bisakah sebuah penduga tetap tak bias meskipun sebarannya sangat lebar?

> *Jawaban Anda:*
>
> ......


---
# Latihan Mandiri 4 — Hukum Bilangan Besar

**Soal asli:** *Pada Bagian 2, naikkan `M` dari 2000 menjadi 10000. Apakah `mean(ŵ)` menjadi lebih dekat ke `w*`? Jelaskan kaitannya dengan hukum bilangan besar.*

### 4.1 Membandingkan M = 2000 dan M = 10000

Lengkapi bagian bertanda `____`. Kali ini σ ditahan tetap 1,0.

In [ ]:
hasil_M = []
for M in [2000, ____]:
    ws, w_star = simulasi_ols(sigma=1.0, M=M)
    rata2 = ws.mean(axis=0)

    # TODO (b): hitung jarak (norma) antara rata2 dan w_star
    jarak = np.linalg.norm(____ - w_star)

    hasil_M.append({"M": M, "rata2_w": np.round(rata2, 5), "jarak_ke_wstar": jarak})
    print(f"M={M:<6}  rata-rata w = {np.round(rata2, 5)}  jarak ke w* = {jarak:.5f}  "
          f"(perkiraan 1/sqrt(M) = {1/np.sqrt(M):.5f})")

### 4.2 Tabel Hasil Pengamatan

| M | Rata-rata $\hat{\mathbf{w}}$ | Jarak ke $\mathbf{w}^*$ |
|---|---|---|
| 2.000 | ...... | ...... |
| 10.000 | ...... | ...... |

### 4.3 Pertanyaan Analisis

**4a.** Apakah jarak ke $\mathbf{w}^*$ mengecil ketika M dinaikkan dari 2.000 menjadi 10.000?

> *Jawaban Anda:*
>
> ......

**4b.** Bandingkan rasio pengecilan jarak yang Anda amati dengan perkiraan teoretis $\sqrt{10000/2000} = \sqrt{5} \approx 2{,}24$ kali. Apakah cukup dekat?

> *Jawaban Anda:*
>
> ......

**4c.** Jelaskan dengan kalimat Anda sendiri: apa perbedaan antara memperbesar `M` (jumlah dataset yang disimulasikan) dengan memperbesar `n` (jumlah titik data pada **satu** dataset, yang dibahas pada Latihan 3 via σ)? Mengapa keduanya sama-sama "membuat hasil lebih presisi" tetapi lewat mekanisme yang berbeda?

> *Jawaban Anda:*
>
> ......


---
# Latihan Mandiri 5 — Selang Kepercayaan untuk Rata-Rata 100 Pengamatan

**Soal asli:** *Pada Bagian 3, ubah `ci_mean10` menjadi rata-rata 100 pengamatan. Berapa kali lipat pita itu menyempit dibandingkan pita satu pengamatan?*

### 5.1 Menghitung ketiga lebar pita

Lengkapi bagian bertanda `____`.

In [ ]:
sigma_asumsi = 0.25

ci_single = 1.96 * sigma_asumsi
ci_mean10 = 1.96 * sigma_asumsi / np.sqrt(10)

# TODO (a): buat ci_mean100 dengan pola yang sama, untuk rata-rata 100 pengamatan
ci_mean100 = 1.96 * sigma_asumsi / np.sqrt(____)

print("Selang 95% satu pengamatan      :", round(ci_single, 4))
print("Selang 95% rata-rata 10 :", round(ci_mean10, 4), " (menyempit", round(ci_single/ci_mean10, 3), "kali)")
print("Selang 95% rata-rata 100:", round(ci_mean100, 4), " (menyempit", round(ci_single/ci_mean100, 3), "kali)")

### 5.2 Tabel Hasil Pengamatan

| Jenis selang | Lebar | Penyempitan dibanding satu pengamatan |
|---|---|---|
| Satu pengamatan | ...... | 1× (acuan) |
| Rata-rata 10 pengamatan | ...... | ...... |
| Rata-rata 100 pengamatan | ...... | ...... |

### 5.3 Pertanyaan Analisis

**5a.** Berapa kali lipat pita menyempit dari "satu pengamatan" ke "rata-rata 100 pengamatan"? Apakah tepat 10 kali?

> *Jawaban Anda:*
>
> ......

**5b.** Bila Anda ingin pitanya menyempit menjadi **20 kali** lebih sempit dari satu pengamatan, berapa banyak pengamatan yang perlu dirata-ratakan?

> *Jawaban Anda:*
>
> ......

**5c.** Mengapa untuk memperoleh penyempitan 10 kali lipat kita butuh **100** pengamatan (bukan 10)? Kaitkan dengan bentuk rumus $1/\sqrt{n}$.

> *Jawaban Anda:*
>
> ......


---
# Latihan Mandiri 6 — Selang Kepercayaan pada Model yang Overfitting

**Soal asli:** *Pada Bagian 3, naikkan derajat polinomial dari 3 menjadi 15. Apakah pita kepercayaan masih menggambarkan ketidakpastian dengan jujur? Kaitkan dengan overfitting.*

### 6.1 Mencocokkan derajat 3 dan derajat 15

Lengkapi bagian bertanda `____`.

In [ ]:
# TODO (a): cocokkan polinomial derajat 15 pada x_demo, t_demo
phi3, w3 = cocokkan_polinomial(x_demo, t_demo, 3)
phi15, w15 = cocokkan_polinomial(x_demo, t_demo, ____)

xg = np.linspace(-1, 1, 300)[:, None]
yg3 = (phi3.transform(xg) @ w3).ravel()
yg15 = (phi15.transform(xg) @ w15).ravel()

# Simpangan baku residual AKTUAL (bukan asumsi sigma=0.25)
resid3 = float((t_demo - phi3.transform(x_demo) @ w3).std())
resid15 = float((t_demo - phi15.transform(x_demo) @ w15).std())

print("Simpangan baku residual aktual, derajat 3 :", round(resid3, 4), " (sigma yang diasumsikan: 0.25)")
print("Simpangan baku residual aktual, derajat 15:", round(resid15, 4), " (sigma yang diasumsikan: 0.25)")
print("\nRentang prediksi derajat 3 :", round(yg3.min(), 3), "sampai", round(yg3.max(), 3))
print("Rentang prediksi derajat 15:", round(yg15.min(), 3), "sampai", round(yg15.max(), 3))

### 6.2 Menggambar kedua kurva beserta pita (lebar pita TETAP, seperti di notebook utama)

In [ ]:
ci = 1.96 * sigma_asumsi   # lebar pita TIDAK berubah menurut derajat -- itulah intinya

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2), sharey=False)
for a, (phi, w, judul) in zip(ax, [(phi3, w3, "Derajat 3"), (phi15, w15, "Derajat 15")]):
    yg = (phi.transform(xg) @ w).ravel()
    a.scatter(x_demo, t_demo, alpha=0.5, s=20)
    a.plot(xg, yg, color="#D55E00", lw=2)
    a.fill_between(xg.ravel(), yg - ci, yg + ci, color="#0072B2", alpha=0.25)
    a.set_title(judul); a.set_xlabel("x"); a.set_ylabel("t")
plt.tight_layout(); plt.show()

### 6.3 Tabel Hasil Pengamatan

| Aspek | Derajat 3 | Derajat 15 |
|---|---|---|
| Simpangan baku residual **aktual** | ...... | ...... |
| σ yang **diasumsikan** rumus pita | 0,25 | 0,25 |
| Rentang nilai prediksi pada $x\in[-1,1]$ | ...... | ...... |

### 6.4 Pertanyaan Analisis

**6a.** Bandingkan simpangan baku residual aktual derajat 15 dengan nilai σ yang diasumsikan (0,25). Apakah keduanya dekat? Apa artinya bagi kecocokan model **pada data latih**?

> *Jawaban Anda:*
>
> ......

**6b.** Perhatikan rentang nilai prediksi derajat 15 — apakah ada nilai yang jauh melebihi rentang wajar target (target berasal dari $\sin$, sehingga secara teori berada sekitar $[-1{,}5,\ 1{,}5]$)? Di bagian grafik sebelah mana (tengah atau tepi) penyimpangan itu terjadi?

> *Jawaban Anda:*
>
> ......

**6c.** Sekalipun residual pada data latih terlihat baik-baik saja, mengapa pita kepercayaan pada derajat 15 **tidak jujur** di daerah tepi? Petunjuk: pita ini dihitung dari σ yang **diasumsikan tetap**, bukan dari seberapa liar kurva tersebut sesungguhnya berperilaku di luar titik-titik data.

> *Jawaban Anda:*
>
> ......


---
# Latihan Mandiri 7 — Ketangguhan terhadap Pencilan: MSE vs MAE

**Soal asli:** *Tambahkan satu pencilan ekstrem pada data Bagian 3 (misalnya $t = 10$ pada suatu $x$), lalu cocokkan ulang. Seberapa jauh kurvanya tertarik? Menurut Anda apa yang akan terjadi bila memakai MAE alih-alih MSE?*

### 7.1 Menambahkan pencilan dan mencocokkan ulang

Lengkapi bagian bertanda `____`. Kita bandingkan tiga kurva: **bersih** (tanpa pencilan), **MSE dengan pencilan**, dan **MAE dengan pencilan**.

In [ ]:
# TODO (a): tambahkan satu titik pencilan pada x=0.5 dengan t=10
x_pencilan = np.vstack([x_demo, [[0.5]]])
t_pencilan = np.vstack([t_demo, [[____]]])

phi_bersih, w_bersih = cocokkan_polinomial(x_demo, t_demo, 3)

# TODO (b): cocokkan MSE (L2) pada data yang SUDAH ada pencilannya
phi_mse, w_mse = cocokkan_polinomial(____, t_pencilan, 3)

# TODO (c): cocokkan MAE (L1) pada data yang sama
phi_mae, w_mae = cocokkan_polinomial_l1(x_pencilan, ____, 3)

xg = np.linspace(-1, 1, 300)[:, None]
y_bersih = (phi_bersih.transform(xg) @ w_bersih).ravel()
y_mse = (phi_mse.transform(xg) @ w_mse).ravel()
y_mae = (phi_mae.transform(xg) @ w_mae).ravel()

print("Pergeseran maksimum MSE (akibat pencilan) :", round(np.max(np.abs(y_mse - y_bersih)), 4))
print("Pergeseran maksimum MAE (akibat pencilan) :", round(np.max(np.abs(y_mae - y_bersih)), 4))
print("Pergeseran rata-rata MSE :", round(np.mean(np.abs(y_mse - y_bersih)), 4))
print("Pergeseran rata-rata MAE :", round(np.mean(np.abs(y_mae - y_bersih)), 4))

### 7.2 Menggambar ketiga kurva

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.scatter(x_demo, t_demo, alpha=0.4, s=20, color="gray", label="Data bersih")
plt.scatter([0.5], [10], color="red", marker="X", s=120, label="Pencilan (x=0,5, t=10)", zorder=5)
plt.plot(xg, y_bersih, "--", color="black", lw=2, label="Model bersih (tanpa pencilan)")
plt.plot(xg, y_mse, color="#D55E00", lw=2, label="MSE dengan pencilan")
plt.plot(xg, y_mae, color="#009E73", lw=2, label="MAE dengan pencilan")
plt.ylim(-2, 4)
plt.xlabel("x"); plt.ylabel("t"); plt.legend(fontsize=9)
plt.title("Pengaruh Satu Pencilan: MSE vs MAE"); plt.show()

### 7.3 Tabel Hasil Pengamatan

| Metrik | MSE (L2) | MAE (L1) |
|---|---|---|
| Pergeseran maksimum akibat pencilan | ...... | ...... |
| Pergeseran rata-rata akibat pencilan | ...... | ...... |

### 7.4 Pertanyaan Analisis

**7a.** Model mana yang bergeser lebih jauh akibat satu pencilan — MSE atau MAE? Sebutkan angkanya.

> *Jawaban Anda:*
>
> ......

**7b.** Perhatikan grafik pada sel 7.2, khususnya di sekitar $x=0{,}5$ (lokasi pencilan). Kurva mana yang "tertarik" paling dekat ke arah pencilan?

> *Jawaban Anda:*
>
> ......

**7c.** Kaitkan hasil ini dengan Bagian 4 notebook utama (teori keputusan): mengapa MSE menghukum pencilan jauh lebih berat daripada MAE? Hubungkan dengan bentuk kuadrat vs bentuk mutlak pada fungsi kerugiannya.

> *Jawaban Anda:*
>
> ......

**7d.** Sebelum menjalankan sel 7.1, soal aslinya meminta Anda **menebak dulu** apa yang akan terjadi bila memakai MAE. Apakah dugaan Anda sesuai dengan hasil yang diperoleh?

> *Jawaban Anda:*
>
> ......


---
# Latihan Mandiri 8 — Membuktikan Kesetaraan Kuadrat Terkecil dan MLE

**Soal asli:** *Buktikan dengan kalimat Anda sendiri: mengapa memaksimalkan $\log L(\mathbf{w})$ menghasilkan titik yang sama dengan meminimalkan $\|\mathbf{y} - \mathbb{X}\mathbf{w}\|^2$?*

Latihan ini murni **konseptual** — tidak ada `TODO` kode untuk dilengkapi. Sel berikut membantu Anda memeriksa jawabannya secara numerik sebelum menuliskan buktinya.

### 8.1 Memeriksa kembali kesetaraan tersebut secara numerik

In [ ]:
rng = np.random.default_rng(SEED)
n, sigma = 80, 1.0
x = np.linspace(0, 5, n)
X = np.c_[x, np.ones_like(x)]
w_star = np.array([2.0, 1.0])
y = X @ w_star + rng.normal(0, sigma, size=n)

def sse(w):
    r = y - X @ w
    return float(r @ r)

def neg_loglik(w):
    return 0.5 * sse(w) / (sigma ** 2)

w_hat = np.linalg.pinv(X) @ y

# Uji beberapa titik w DI SEKITAR w_hat: apakah keduanya sama-sama minimum di sana?
for w_uji in [w_hat, w_hat + np.array([0.1, 0]), w_hat + np.array([0, 0.2]), w_hat - np.array([0.05, 0.05])]:
    print(f"w={np.round(w_uji,3)}  SSE={sse(w_uji):8.3f}   -logL={neg_loglik(w_uji):8.3f}   "
          f"rasio SSE/-logL={sse(w_uji)/neg_loglik(w_uji):.4f}")

### 8.2 Tabel Hasil Pengamatan

| Titik $\mathbf{w}$ | SSE | $-\log L$ | Rasio SSE / $-\log L$ |
|---|---|---|---|
| $\hat{\mathbf{w}}$ (optimum) | ...... | ...... | ...... |
| Bergeser sedikit (mana pun) | ...... | ...... | ...... |

### 8.3 Pertanyaan Analisis

**8a.** Perhatikan kolom rasio SSE/$-\log L$ pada tabel Anda. Apakah rasionya **sama** untuk semua titik $\mathbf{w}$ yang diuji? Angka berapa itu, dan cocokkah dengan $2\sigma^2$?

> *Jawaban Anda:*
>
> ......

**8b.** Karena rasio kedua fungsi itu **konstan** di semua titik (bukan hanya di titik optimum), apa yang dapat Anda simpulkan tentang hubungan antara $\arg\min \text{SSE}(\mathbf{w})$ dan $\arg\min (-\log L(\mathbf{w}))$?

> *Jawaban Anda:*
>
> ......

**8c.** Tuliskan jawaban akhir Anda untuk soal Latihan Mandiri 8 dalam 3–5 kalimat. Sertakan langkah aljabar singkat: mulai dari $\log L(\mathbf{w}) = -\frac{1}{2\sigma^2}\|\mathbf{y}-\mathbb{X}\mathbf{w}\|^2 + \text{konstanta}$, lalu jelaskan mengapa mengalikan dengan bilangan positif dan menambah konstanta tidak mengubah letak titik optimumnya.

> *Jawaban Anda:*
>
> ......


---
# Kesimpulan dan Refleksi

### Kesimpulan

*Tuliskan minimal delapan poin kesimpulan — satu untuk setiap latihan mandiri — berdasarkan hasil percobaan Anda sendiri.*

1. (Kolinearitas vs bilangan kondisi) ......
2. (λ terbaik untuk Ridge) ......
3. (σ vs sebaran penduga) ......
4. (Hukum bilangan besar) ......
5. (Selang kepercayaan rata-rata 100) ......
6. (Pita kepercayaan vs overfitting) ......
7. (MSE vs MAE terhadap pencilan) ......
8. (Kesetaraan kuadrat terkecil dan MLE) ......

### Refleksi

**R1.** Latihan mana yang paling mengubah pemahaman Anda tentang regresi probabilistik? Jelaskan.

> *Jawaban Anda:*
>
> ......

**R2.** Error apa yang Anda temui saat mengerjakan, dan bagaimana Anda mengatasinya?

> *Jawaban Anda:*
>
> ......

**R3.** Bila diberi dataset nyata yang diduga mengandung pencilan, langkah apa saja yang akan Anda ambil berdasarkan pengalaman mengerjakan lembar kerja ini?

> *Jawaban Anda:*
>
> ......


---
# Rubrik Penilaian

*Bagian ini diisi oleh dosen atau asisten praktikum.*

| No | Aspek yang Dinilai | Bobot | Skor (0–100) | Nilai |
|---|---|---|---|---|
| 1 | Kelengkapan tabel hasil pengamatan (Latihan 1–8) | 20% | | |
| 2 | Ketepatan pengisian kode bertanda `TODO` | 25% | | |
| 3 | Kualitas jawaban analisis (1a–8c) | 35% | | |
| 4 | Kesimpulan dan refleksi | 20% | | |
| | **Nilai Akhir** | **100%** | | |

**Catatan dosen:**

> ......

### Pedoman skor jawaban analisis

| Skor | Kriteria |
|---|---|
| 85–100 | Jawaban tepat, didukung angka dari percobaan sendiri, dan menunjukkan penalaran yang jelas |
| 70–84 | Jawaban tepat tetapi kurang didukung data atau penalarannya dangkal |
| 55–69 | Jawaban sebagian benar, atau hanya mengulang teori tanpa mengaitkan hasil percobaan |
| < 55 | Jawaban tidak tepat, kosong, atau merupakan salinan dari mahasiswa lain |


---
# Daftar Periksa Sebelum Mengumpulkan

Centang dengan mengganti `[ ]` menjadi `[x]` (klik dua kali sel ini untuk mengedit).

- [ ] Identitas pada bagian atas sudah diisi lengkap
- [ ] `NIM_3_DIGIT` sudah diganti dengan NIM saya sendiri
- [ ] Semua sel bertanda `TODO` sudah dilengkapi dan berjalan tanpa error
- [ ] Seluruh sel sudah dijalankan berurutan dari atas ke bawah
- [ ] Semua tabel hasil pengamatan sudah diisi angka dari komputer saya
- [ ] Semua pertanyaan analisis 1a sampai 8c sudah dijawab
- [ ] Kesimpulan (8 poin) dan refleksi R1–R3 sudah ditulis
- [ ] Notebook sudah disimpan (`Ctrl + S`) sebelum diekspor

### Langkah ekspor PDF

1. Simpan notebook: `Ctrl + S`
2. `File` → `Save and Export Notebook As...` → `HTML`
3. Buka berkas HTML di browser, tekan `Ctrl + P`, pilih **Save as PDF**
4. Pada dialog cetak, aktifkan **Background graphics** agar grafik ikut tercetak berwarna
5. Beri nama berkas: `LKM_Latihan08_NIM_NamaLengkap.pdf`

---

*Lembar Kerja Mahasiswa — Latihan Mandiri untuk Pertemuan08_Regresi_Linear_3_Probabilistik.ipynb*
*Materi diadaptasi dari kuliah "Linear Regression" oleh Joseph E. Gonzalez dan Narges Norouzi.*
